# 보호색 동물 YOLO 학습

런타임 → 런타임 유형 변경 → GPU를 선택하세요. COD10K/MoCA 원본은 직접 내려받고 주석을 확인합니다. 이 노트북은 자동 데이터 다운로드나 추적 ID 추정을 하지 않습니다. 실제 학습·실영상 성능은 아직 미실행입니다.


In [ ]:
from pathlib import Path
import subprocess

project = Path("/content/AI_Project_Level_3-main")
if not project.exists():
    subprocess.run(["git", "clone", "--branch", "feature/camouflage", "https://github.com/kumohyunho/AI_Project_Level_3-main.git", str(project)], check=True)
%cd /content/AI_Project_Level_3-main
%pip install -r requirements.txt


## 원본 데이터 준비

`docs/CAMOUFLAGE.md`의 폴더와 CSV 형식을 확인하고 `datasets/raw/` 아래에 데이터를 준비하세요.
- COD10K: `datasets/raw/COD10K-v3/{Train,Test}/{Image,GT_Object}`
- MoCA: `datasets/raw/MoCA/frames/<영상>/00000.jpg`와 `boxes.csv`
- MoCA CSV에 개체 ID가 없다면 검출 학습만 가능합니다. 영상마다 같은 동물 한 마리임을 직접 확인했을 때만 `--single-object`를 추가하세요.
- `--fps`는 실제 추출 프레임 FPS로 수정하세요. 영상을 학습·검증·테스트에 걸쳐 나누면 안 됩니다.


In [ ]:
import subprocess

subprocess.run([
    "python", "camouflage_dataset.py",
    "--cod10k", "datasets/raw/COD10K-v3",
    "--moca-frames", "datasets/raw/MoCA/frames",
    "--moca-csv", "datasets/raw/MoCA/boxes.csv",
    "--fps", "24", "--output", "datasets/camouflage",
], check=True)


## 추가 학습과 설치

GPU 메모리가 부족하면 batch를 낮추세요. 기존 camouflage.pt/.json은 함께 백업한 뒤 실행합니다. test 결과는 모델 선택이나 파라미터 조정에 사용하지 않습니다.


In [ ]:
subprocess.run([
    "python", "train_camouflage.py",
    "--data", "datasets/camouflage/dataset.yaml",
    "--weights", "yolo11n.pt", "--epochs", "50",
    "--imgsz", "640", "--batch", "8", "--device", "0", "--install",
], check=True)


## 추적 검증 (개체 ID 정답이 확인된 영상만)

처음에는 val에서 설정을 검토하고, 최종 설정을 고른 뒤 test를 실행합니다. 희소 주석은 해당 프레임만 평가하며, 미주석 프레임도 추적은 계속합니다. 아래 셀은 ID 정답이 없으면 중단됩니다.


In [ ]:
subprocess.run([
    "python", "evaluate_camouflage.py", "datasets/camouflage/tracking/val",
    "--device", "0", "--output", "results/camouflage/val",
], check=True)
# 최종 설정 확정 후에만 실행:
# subprocess.run(["python", "evaluate_camouflage.py", "datasets/camouflage/tracking/test",
#                 "--device", "0", "--output", "results/camouflage/test"], check=True)


## Google Drive에 결과 저장

가중치와 설정은 반드시 함께 보관하세요. 내 PC 프로젝트의 같은 위치에 복사하고 GUI에서 보호색 동물 확장을 켭니다. 기존 Drive 결과 폴더는 덮어쓰지 않습니다.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone
import shutil

drive.mount("/content/drive")
output = Path("/content/drive/MyDrive/AI_Project_Level_3_outputs") / ("camouflage_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S"))
output.mkdir(parents=True, exist_ok=False)
(output / "models").mkdir()
for name in ("camouflage.pt", "camouflage.json"):
    shutil.copy2(project / "models" / name, output / "models" / name)
shutil.copy2(project / "datasets/camouflage/splits.json", output / "splits.json")
results = project / "results/camouflage"
if results.exists():
    shutil.copytree(results, output / "results/camouflage")
print("저장 위치:", output)
